In [13]:
%reset -f

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

In [2]:
df_train = pd.read_csv('../input/train_set.csv', sep="\t")
df_test = pd.read_csv('../input/test_set.csv', sep="\t")

# Word2Vec Method

In [3]:
from gensim.models import Word2Vec

C:\Users\lsaid\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [4]:
corpus = pd.concat([ df_train["reviewText"], df_test["reviewText"] ], axis=0)
corpus

0         ['sexi', 'read', 'am', 'sucker', 'for', 'the',...
1         ['love', 'be', 'abl', 'to', 'read', 'all', 'th...
2         ['like', 'thi', 'stori', 'but', 'aggrav', 'me'...
3         ['didn', 'like', 'the', 'wai', 'it', 'end', 'w...
4         ['shorter', 'than', 'like', 'but', 'full', 'of...
                                ...                        
290316    ['after', 'have', 'read', 'book', 'in', 'thi',...
290317                                               ['ok']
290318    ['davina', 'is', 'gone', 'mourn', 'her', 'alon...
290319                                             ['okai']
290320    ['unbeliev', 'how', 'thi', 'writer', 'can', 'g...
Name: reviewText, Length: 879758, dtype: object

In [9]:
corpus.shape

(879758,)

In [6]:
model_w2v = Word2Vec(sentences=corpus, vector_size=200, window=5, min_count=1, workers=4)

In [7]:
model_w2v.save("../models/word2vec_model")

In [8]:
model_w2v = Word2Vec.load("../models/word2vec_model")

In [10]:
print(len(model_w2v.wv))

32


In [11]:
model_vector = (np.mean([model_w2v.wv[token] for token in df_train['reviewText'][500]], axis=0)).tolist()

In [23]:
model_vector

[-0.3165256977081299,
 0.41685688495635986,
 0.44521811604499817,
 -0.25018417835235596,
 -0.1872001737356186,
 -0.21304889023303986,
 0.033861104398965836,
 -0.7244157791137695,
 0.1686595380306244,
 0.6376625895500183,
 -0.24917171895503998,
 0.4673357605934143,
 -0.08399970829486847,
 -0.3022077679634094,
 -0.14340075850486755,
 -0.19013001024723053,
 0.07175645977258682,
 -0.594476044178009,
 0.16188284754753113,
 -0.07785817235708237,
 -0.28139829635620117,
 0.13655024766921997,
 -0.32496532797813416,
 0.10248252004384995,
 0.046523626893758774,
 0.2573031485080719,
 0.07071355730295181,
 0.5808283090591431,
 0.466829389333725,
 -0.18793724477291107,
 0.042511045932769775,
 -0.03380060940980911,
 0.23705120384693146,
 -0.1644241213798523,
 0.054400037974119186,
 0.6449006199836731,
 -0.290507048368454,
 0.35937392711639404,
 0.14625686407089233,
 0.03820338472723961,
 0.3934827446937561,
 -0.40134623646736145,
 0.12339848279953003,
 0.1375758945941925,
 -0.2145184427499771,
 -0.57

In [13]:
len(model_vector)

200

In [14]:
X_vector_train = []
for review in df_train['reviewText']:
    vector = (np.mean([model_w2v.wv[token] for token in review], axis=0)).tolist()
    X_vector_train.append(vector)

In [25]:
X_vector_train = np.array(X_vector_train)

In [15]:
y_train = df_train["overall"].values

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

from sklearn.metrics import accuracy_score,make_scorer, classification_report
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.layers import Embedding

In [17]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

In [27]:
from sklearn.tree import DecisionTreeClassifier
#Initialize the model
clf_decision = DecisionTreeClassifier()
clf_decision.fit(X_vector_train, y_train_encoded)

DecisionTreeClassifier()

In [28]:
from sklearn.metrics import classification_report
X_vector_test = []
for review in df_test['reviewText']:
    vector = (np.mean([model_w2v.wv[token] for token in review], axis=0)).tolist()
    X_vector_test.append(vector)

In [29]:
test_predictions_word2vec = clf_decision.predict(X_vector_test)
print(classification_report(df_test['overall'],test_predictions_word2vec))

C:\Users\lsaid\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\lsaid\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\lsaid\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\lsaid\anaconda3\Lib\site-packages\sklearn\metrics\_cla

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.05      0.05      0.05      7035
           2       0.05      0.16      0.08      7952
           3       0.09      0.24      0.13     24827
           4       0.21      0.58      0.31     65927
           5       0.00      0.00      0.00    184580

    accuracy                           0.16    290321
   macro avg       0.07      0.17      0.09    290321
weighted avg       0.06      0.16      0.08    290321



C:\Users\lsaid\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\lsaid\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [38]:
model = Sequential()
model.add(Dense(128, input_shape=(X_vector_train.shape[1],), activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(5, activation='softmax'))  # 5 classes for ratings 1 to 5
optimizer = Adam(learning_rate=0.001)
loss = SparseCategoricalCrossentropy()
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

In [39]:
# Training the model
model.fit(X_vector_train, y_train_encoded, epochs=5, batch_size=32, validation_split=0.1)

Epoch 1/5
16578/16578 [==============================] - 53s 3ms/step - loss: 1.0116 - accuracy: 0.6375 - val_loss: 1.0125 - val_accuracy: 0.6345
Epoch 2/5
16578/16578 [==============================] - 52s 3ms/step - loss: 0.9972 - accuracy: 0.6379 - val_loss: 0.9900 - val_accuracy: 0.6333
Epoch 3/5
16578/16578 [==============================] - 53s 3ms/step - loss: 0.9929 - accuracy: 0.6379 - val_loss: 0.9876 - val_accuracy: 0.6345
Epoch 4/5
16578/16578 [==============================] - 53s 3ms/step - loss: 0.9908 - accuracy: 0.6379 - val_loss: 0.9905 - val_accuracy: 0.6345
Epoch 5/5
16578/16578 [==============================] - 53s 3ms/step - loss: 0.9891 - accuracy: 0.6380 - val_loss: 0.9853 - val_accuracy: 0.6345


In [30]:
X_test = []
for review in df_test['reviewText']:
    vector = (np.mean([model_w2v.wv[token] for token in review], axis=0)).tolist()
    X_test.append(vector)
X_test = np.array(X_test)

In [31]:
y_test = df_test["overall"].values

In [40]:

# Evaluating the model
y_pred_proba = model.predict(np.array(X_test))
y_pred = label_encoder.inverse_transform(y_pred_proba.argmax(axis=1))

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

# Additional metrics
print(classification_report(y_test, y_pred))

9073/9073 [==============================] - 14s 2ms/step
Accuracy: 0.64


C:\Users\lsaid\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\lsaid\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

           1       0.25      0.00      0.00      7035
           2       0.00      0.00      0.00      7952
           3       0.54      0.02      0.04     24827
           4       0.00      0.00      0.00     65927
           5       0.64      1.00      0.78    184580

    accuracy                           0.64    290321
   macro avg       0.28      0.20      0.16    290321
weighted avg       0.46      0.64      0.50    290321



C:\Users\lsaid\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# TF-IDF Method

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [11]:
tfidf = TfidfVectorizer(max_features=1000)
X = tfidf.fit_transform(corpus).toarray()
y = df["overall"].values

In [12]:
from sklearn.model_selection import train_test_split

X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test = train_test_split(X, y, test_size=0.33, random_state=50)

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

from sklearn.metrics import accuracy_score,make_scorer, classification_report
from sklearn.model_selection import GridSearchCV


In [14]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_tfidf_train_encoded = label_encoder.fit_transform(y_tfidf_train)

In [15]:
model = Sequential()
model.add(Dense(128, input_shape=(X_tfidf_train.shape[1],), activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(5, activation='softmax'))  # 5 classes for ratings 1 to 5
optimizer = Adam(learning_rate=0.001)
loss = SparseCategoricalCrossentropy()
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

# Training the model
model.fit(X_tfidf_train, y_tfidf_train_encoded, epochs=5, batch_size=32, validation_split=0.1)

# Evaluating the model
y_pred_proba = model.predict(X_tfidf_test)
y_pred = label_encoder.inverse_transform(y_pred_proba.argmax(axis=1))

# Evaluation
accuracy = accuracy_score(y_tfidf_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

# Additional metrics
print(classification_report(y_tfidf_test, y_pred))



Epoch 1/5


16578/16578 [==============================] - 60s 4ms/step - loss: 0.7815 - accuracy: 0.6800 - val_loss: 0.7682 - val_accuracy: 0.6838
Epoch 2/5
16578/16578 [==============================] - 57s 3ms/step - loss: 0.7564 - accuracy: 0.6894 - val_loss: 0.7588 - val_accuracy: 0.6860
Epoch 3/5
16578/16578 [==============================] - 57s 3ms/step - loss: 0.7431 - accuracy: 0.6945 - val_loss: 0.7568 - val_accuracy: 0.6874
Epoch 4/5
16578/16578 [==============================] - 57s 3ms/step - loss: 0.7325 - accuracy: 0.6991 - val_loss: 0.7566 - val_accuracy: 0.6879
Epoch 5/5
9073/9073 [==============================] - 17s 2ms/step
Accuracy: 0.69
              precision    recall  f1-score   support

           1       0.51      0.38      0.44      7035
           2       0.38      0.11      0.17      7952
           3       0.46      0.37      0.41     24827
           4       0.49      0.22      0.30     65927
           5       0.74      0.94      0.83    184580

    

# Single prediction 

In [16]:
def predict_rating(text):
    corpus = []
    res = re.sub('[^a-zA-Z]', ' ', text.lower())
    res = res.split()
    lemmatizer = WordNetLemmatizer()
    all_stopwords = stopwords.words('english')
    all_stopwords.remove("not")
    res = [lemmatizer.lemmatize(word) for word in res if not word in set(all_stopwords)]
    res = ' '.join(res)
    corpus.append(res)
    X = tfidf.transform(corpus).toarray()  # Use transform, not fit_transform
    prediction_proba = model.predict(X)
    predicted_class = label_encoder.inverse_transform([prediction_proba.argmax()])[0]
    print(f'Predicted Rating: {predicted_class}')

In [22]:
text = "i have seen better."
predict_rating(text)

1/1 [==============================] - 0s 23ms/step
Predicted Rating: 5


As you can see, some sentences are not well guess. It might be because of :
- Lack of datas on low rating 
- Use of TF-IDF that doesn't take the context of words into account

So the next steps should be :

- Getting more data of bad rated reviews (1 to 3)
- Learn about Contextual Word Embedding